---Decision Trees---

Applying label encoding

In [36]:
df = pd.read_csv("college_iot_energy_monitoring.csv")

# 2. Encode categorical columns
label_encoders = {}

categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in categorical_cols:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col].astype(str))
    label_encoders[col] = encoder

# 3. Separate features and target
X = df.drop(columns=["energy"])
y = df["energy"]

In [29]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Define the decision tree
tree = DecisionTreeRegressor(random_state=42)

# 6. Tune hyperparameters
param_grid = {
    "max_depth": [3, 5, 8, 10, 15, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["squared_error", "absolute_error"]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=tree,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1
)

# 7. Train and tune
search.fit(X_train, y_train)

best_tree = search.best_estimator_
y_pred = best_tree.predict(X_test)

# 8. Evaluate
mse = mean_squared_error(y_test, y_pred)

print("Best Parameters:", search.best_params_)
print("Best CV RMSE:", np.sqrt(-search.best_score_))
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test MSE:", mse)
print("Test RMSE:", np.sqrt(mse))
print("Test R2:", r2_score(y_test, y_pred))

# 9. Show encoded category mappings
for col, encoder in label_encoders.items():
    print(f"\n{col} encoding:")
    print(dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))


Best Parameters: {'criterion': 'squared_error', 'max_depth': 8, 'min_samples_leaf': 5, 'min_samples_split': 2}
Best CV RMSE: 0.013628001376588096
Test MAE: 0.00894166043619019
Test MSE: 0.0001409140600082199
Test RMSE: 0.01187072280900451
Test R2: 0.942898617451785

device encoding:
{'Agricultural Sensor': np.int64(0), 'EV Charger': np.int64(1), 'Edge Gateway': np.int64(2), 'Health Monitor': np.int64(3), 'Industrial Sensor': np.int64(4), 'Smart Camera': np.int64(5), 'Smart Lighting': np.int64(6), 'Smart Meter': np.int64(7), 'Smart Thermostat': np.int64(8), 'Wearable Device': np.int64(9)}


In [30]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load the feature-engineered dataset


# 2. Identify categorical columns
categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# 3. Separate features and target
X = df.drop(columns=["energy"]).copy()
y = df["energy"].copy()

# 4. Split BEFORE encoding to avoid fitting encoders on test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Fit LabelEncoders on training data only
label_encoders = {}

for col in categorical_cols:
    encoder = LabelEncoder()
    X_train[col] = encoder.fit_transform(X_train[col].astype(str))
    label_encoders[col] = encoder

    # Handle categories not seen during training
    mapping = {
        category: index
        for index, category in enumerate(encoder.classes_)
    }
    X_test[col] = (
        X_test[col].astype(str).map(mapping).fillna(-1).astype(int)
    )

# 6. Define the Decision Tree Regressor
tree = DecisionTreeRegressor(random_state=42)

# 7. Define hyperparameter combinations
param_grid = {
    "criterion": ["squared_error", "absolute_error"],
    "max_depth": [3, 5, 8, 10, 15, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": [None, "sqrt"]
}

# 8. Set up 5-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=tree,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# 9. Train and tune
search.fit(X_train, y_train)

best_tree = search.best_estimator_
y_pred = best_tree.predict(X_test)

# 10. Evaluate the selected model
mse = mean_squared_error(y_test, y_pred)

print("\nBest Hyperparameters:")
print(search.best_params_)

print("\nBest Cross-Validation RMSE:")
print(np.sqrt(-search.best_score_))

print("\nTest Set Metrics:")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mse)
print("RMSE:", np.sqrt(mse))
print("R2 Score:", r2_score(y_test, y_pred))

# 11. Compare actual and predicted energy
results = pd.DataFrame({
    "Actual Energy": y_test.values,
    "Predicted Energy": y_pred
})
print("\nSample Predictions:")
print(results.head(10))


Fitting 5 folds for each of 384 candidates, totalling 1920 fits

Best Hyperparameters:
{'criterion': 'squared_error', 'max_depth': 8, 'max_features': None, 'min_samples_leaf': 5, 'min_samples_split': 2}

Best Cross-Validation RMSE:
0.013628001376588096

Test Set Metrics:
MAE: 0.00894166043619019
MSE: 0.0001409140600082199
RMSE: 0.01187072280900451
R2 Score: 0.942898617451785

Sample Predictions:
   Actual Energy  Predicted Energy
0       0.005602          0.025315
1       0.000264          0.005123
2       0.006491          0.004491
3       0.027863          0.019913
4       0.020342          0.005123
5       0.006538          0.001140
6       0.000000          0.005123
7       0.000000          0.006148
8       0.195922          0.184135
9       0.039902          0.023659


Tree Pruning

In [31]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)



X = df.drop(columns=["energy"]).copy()
y = df["energy"].copy()

# 2. Split data BEFORE encoding categorical features
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 3. Label-encode categorical columns using training data only
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns

for col in categorical_cols:
    encoder = LabelEncoder()

    X_train[col] = encoder.fit_transform(X_train[col].astype(str))

    mapping = {
        category: index
        for index, category in enumerate(encoder.classes_)
    }

    X_test[col] = (
        X_test[col].astype(str).map(mapping).fillna(-1).astype(int)
    )

# 4. Evaluate the unpruned decision tree
baseline = DecisionTreeRegressor(random_state=42)
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

def evaluate_model(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)

    print(f"\n{name}")
    print("MAE :", round(mean_absolute_error(y_true, y_pred), 5))
    print("MSE :", round(mse, 5))
    print("RMSE:", round(np.sqrt(mse), 5))
    print("R2  :", round(r2_score(y_true, y_pred), 5))

evaluate_model("Unpruned Decision Tree", y_test, baseline_pred)

print("Unpruned tree depth:", baseline.get_depth())
print("Unpruned leaf nodes:", baseline.get_n_leaves())

# 5. Find candidate pruning strengths
path = DecisionTreeRegressor(
    random_state=42
).cost_complexity_pruning_path(X_train, y_train)

alphas = np.unique(path.ccp_alphas)

# Exclude the largest alpha, which often produces a root-only tree
if len(alphas) > 1:
    alphas = alphas[:-1]

# Limit candidates to make cross-validation practical
if len(alphas) > 50:
    alphas = np.unique(
        np.quantile(alphas, np.linspace(0, 1, 50))
    )

# 6. Tune ccp_alpha using 5-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid={"ccp_alpha": alphas},
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1
)

search.fit(X_train, y_train)

# 7. Evaluate the best pruned tree on the test set
pruned_tree = search.best_estimator_
pruned_pred = pruned_tree.predict(X_test)

print("\nBest pruning alpha:", search.best_params_["ccp_alpha"])
print("Best CV RMSE:", round(np.sqrt(-search.best_score_), 5))

evaluate_model("Pruned Decision Tree", y_test, pruned_pred)

print("Pruned tree depth:", pruned_tree.get_depth())
print("Pruned leaf nodes:", pruned_tree.get_n_leaves())



Unpruned Decision Tree
MAE : 0.01119
MSE : 0.00025
RMSE: 0.01589
R2  : 0.89774
Unpruned tree depth: 23
Unpruned leaf nodes: 1381

Best pruning alpha: 6.687855450221336e-07
Best CV RMSE: 0.01329

Pruned Decision Tree
MAE : 0.00914
MSE : 0.00017
RMSE: 0.01311
R2  : 0.93034
Pruned tree depth: 9
Pruned leaf nodes: 57


---Random Forest---

In [32]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)



# 2. Label-encode categorical columns
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in categorical_cols:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col].astype(str))
    label_encoders[col] = encoder

# 3. Separate features and target
X = df.drop(columns=["energy"])
y = df["energy"]

# 4. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Create Random Forest model
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)

# 6. Train the model
model.fit(X_train, y_train)

# 7. Predict energy consumption
y_pred = model.predict(X_test)

# 8. Evaluate model
mse = mean_squared_error(y_test, y_pred)

print("Random Forest Regression")
print("-------------------------")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mse)
print("RMSE:", np.sqrt(mse))
print("R2 Score:", r2_score(y_test, y_pred))

# 9. Show actual vs predicted
results = pd.DataFrame({
    "Actual Energy": y_test.values,
    "Predicted Energy": y_pred
})

print("\nSample Predictions:")
print(results.head(10))

# 10. Feature importance
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 10 Important Features:")
print(importance.head(10))


Random Forest Regression
-------------------------
MAE: 0.008009132707812098
MSE: 0.00010199789055533994
RMSE: 0.010099400504749771
R2 Score: 0.9586682793230733

Sample Predictions:
   Actual Energy  Predicted Energy
0       0.005602          0.014335
1       0.000264          0.005354
2       0.006491          0.007126
3       0.027863          0.022776
4       0.020342          0.007166
5       0.006538          0.011224
6       0.000000          0.005075
7       0.000000          0.004590
8       0.195922          0.184377
9       0.039902          0.026920

Top 10 Important Features:
power          0.619206
duration       0.364999
humidity       0.004055
temperature    0.003974
day            0.003381
hour           0.003040
device         0.001345
dtype: float64


---hyperparameter tuning---

In [33]:

from sklearn.model_selection import GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [1.0, "sqrt"]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=1
)

search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

best_rf = search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)

mse = mean_squared_error(y_test, y_pred_tuned)

print("Best Parameters:", search.best_params_)
print("Best CV RMSE:", np.sqrt(-search.best_score_))
print("Test MAE:", mean_absolute_error(y_test, y_pred_tuned))
print("Test RMSE:", np.sqrt(mse))
print("Test R2:", r2_score(y_test, y_pred_tuned))


Fitting 5 folds for each of 144 candidates, totalling 720 fits
Best Parameters: {'max_depth': 10, 'max_features': 1.0, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Best CV RMSE: 0.010634925781810535
Test MAE: 0.007899978651089719
Test RMSE: 0.009930102468012358
Test R2: 0.960042366826604


---Training XGBoost---

In [34]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from xgboost import XGBRegressor



# 4. Encode categorical columns
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns

for col in categorical_cols:
    encoder = LabelEncoder()

    X_train[col] = encoder.fit_transform(
        X_train[col].astype(str)
    )

    mapping = {
        category: index
        for index, category in enumerate(encoder.classes_)
    }

    X_test[col] = (
        X_test[col].astype(str)
        .map(mapping)
        .fillna(-1)
        .astype(int)
    )

# 5. Create XGBoost regressor
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

# 6. Train model
xgb_model.fit(X_train, y_train)

# 7. Predict
y_pred = xgb_model.predict(X_test)

# 8. Evaluate
mse = mean_squared_error(y_test, y_pred)

print("XGBoost Regression Results")
print("--------------------------")
print("MAE :", round(mean_absolute_error(y_test, y_pred), 5))
print("MSE :", round(mse, 5))
print("RMSE:", round(np.sqrt(mse), 5))
print("R2  :", round(r2_score(y_test, y_pred), 5))

# 9. Feature importance
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nFeature Importance:")
print(importance)


XGBoost Regression Results
--------------------------
MAE : 0.00768
MSE : 9e-05
RMSE: 0.00974
R2  : 0.96153

Feature Importance:
       Feature  Importance
5        power    0.358446
0       device    0.344889
6     duration    0.256143
1         hour    0.028231
2          day    0.005300
4     humidity    0.003679
3  temperature    0.003312


---Hyperperameter Tuning of XGBoost---

In [35]:

from sklearn.model_selection import GridSearchCV, KFold
from xgboost import XGBRegressor
import numpy as np

param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [3, 4, 6],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

xgb = XGBRegressor(
    objective="reg:squarederror",
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=1
)

search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

best_xgb = search.best_estimator_
tuned_pred = best_xgb.predict(X_test)

print("Best Parameters:", search.best_params_)
print("Best CV RMSE:", round(np.sqrt(-search.best_score_), 5))

mse = mean_squared_error(y_test, tuned_pred)

print("\nTuned XGBoost Results")
print("MAE :", round(mean_absolute_error(y_test, tuned_pred), 5))
print("MSE :", round(mse, 5))
print("RMSE:", round(np.sqrt(mse), 5))
print("R2  :", round(r2_score(y_test, tuned_pred), 5))


Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Best Parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 300, 'subsample': 0.8}
Best CV RMSE: 0.01019

Tuned XGBoost Results
MAE : 0.00774
MSE : 0.0001
RMSE: 0.00992
R2  : 0.96016
